# Universal portfolio - Cover 1996

This is an example of how to evaluate a Universal portfolio. For more details see the library documentation 
https://azapy.readthedocs.io/en/latest/.

We start by importing **azapy** and other useful packages
(**azapy** version must be 1.2.1 or greater).

In [4]:
import azapy as az

print(f"azapy version {az.version()} >= 1.2.0")

azapy version 1.2.6 >= 1.2.0


### Collect historical market data

- `symb` is the list of stock symbols (portfolio components).
- `sdate` and `edate` are the start and end dates of historical time-series.
- `mktdir` is the name of the directory used as a buffer for market data collected from the data provider (in this case _alphavantage_).
    
> Note: if the flag `force=False` then a reading from `dir=mktdir` is attempted. If it fails, then the data provider servers will be accessed. The new data will be saved to the `dir=mktdir`. For more information see the readMkT documentation https://azapy.readthedocs.io/en/latest/.

In [5]:
mktdir = '../MkTdata'
sdate = '2012-01-01'
edate = 'today'
symb = ['VHT', 'SPY', 'XLV', 'GLD', 'ONEQ']

mktdata = az.readMkT(symb, sdate=sdate, edate=edate, file_dir=mktdir)

read VHT data from file
read SPY data from file
read XLV data from file
read GLD data from file
read ONEQ data from file

Request between 2012-01-03 : 2026-05-05
                    VHT         SPY         XLV         GLD        ONEQ
source            yahoo       yahoo       yahoo       yahoo       yahoo
force             False       False       False       False       False
save               True        True        True        True        True
file_dir     ../MkTdata  ../MkTdata  ../MkTdata  ../MkTdata  ../MkTdata
file_format         csv         csv         csv         csv         csv
api_key            None        None        None        None        None
nrow               3605        3605        3605        3605        3605
sdate        2012-01-03  2012-01-03  2012-01-03  2012-01-03  2012-01-03
edate        2026-05-05  2026-05-05  2026-05-05  2026-05-05  2026-05-05
error                No          No          No          No          No
extraction time 0.279 s


### Setup the Universal portfolio

First line is the constructor
Second line sets the model and execute the core computations. It is a Monte Carlo based evaluation.

- `mc_paths` - are the number of simulations per batch (must be >= 1)
- `nr_batches` - are the number of batches (must be >= 1). Note that the computation is multithreaded with one batch per thread.
- `variance_reduction = True` - default value (the MC will use the antithetic variance reduce implied by the permutations of the basket components)
- `alpha_dirichlet = None` - default value. In this case a uniform random generator of vectors in the M-simplex is used. This is equivalent to a Flat Dirichlet random generator (all alpha set to 1).

>Note the the total number of MC simulations is `mc_paths * nr_batches * M!` where `M` is the number of portfolio components and `M!` its factorial.

In [6]:
p4 = az.Port_Universal(mktdata, pname='UnivPort')    
port4 = p4.set_model(mc_paths=100, nr_batches=20, verbose=True)   

nr simulations: 240000
simulation time: 0.225531


### Historical portfolio weights

>Note: if `_CASH_` is not an explicit component of the portfolio (it is not present in the `makdata`), then its weight is
set to `0`.
A `_CASH_` asset can be added to the `mktdata` by using the helper function `azapy.add_cash_security(mktdata)`.

In [7]:
p4.get_weights()

,Droll,Dfix,GLD,ONEQ,SPY,VHT,XLV,_CASH_
0,2015-06-25,2015-06-24,0.200000,0.200000,0.200000,0.200000,0.200000,0
1,2015-09-25,2015-09-24,0.201855,0.199825,0.199666,0.199324,0.199328,0
2,2015-12-28,2015-12-24,0.198453,0.200917,0.200887,0.199667,0.200076,0
3,2016-03-28,2016-03-24,0.203406,0.199633,0.201226,0.197359,0.198375,0
4,2016-06-27,2016-06-24,0.205168,0.198450,0.200392,0.197561,0.198430,0
5,2016-09-27,2016-09-26,0.203634,0.200263,0.200429,0.197612,0.198061,0
6,2016-12-27,2016-12-23,0.199020,0.202654,0.203369,0.197230,0.197726,0
7,2017-03-28,2017-03-27,0.200035,0.202541,0.202225,0.197429,0.197770,0
8,2017-06-27,2017-06-26,0.197942,0.203043,0.201925,0.198445,0.198645,0
9,2017-09-26,2017-09-25,0.198769,0.202867,0.201984,0.198092,0.198288,0


### Portfolio performance view

In [8]:
_ = p4.port_view(fancy=True)

### Portfolio and its components relative performances 

In [9]:
_ = p4.port_view_all(fancy=True)

## Portfolio performance in terms of total return, maximum drawdown, and RoMaD 

- `RR` - total rate of return
- `DD` - maximum drawdown
- `RoMaD` - return over maximum drawdown (`RR/DD`)
- `DD_date` - maximum drawdown date
- `DD_start` - maximum drawdate stating date
- `DD_end` - maximum drawdown ending date

In [10]:
p4.port_perf(fancy=True)

,RR,DD,RoMaD,DD_date,DD_start,DD_end,DD_days
symbol,,,,,,,
UnivPort,12.36,-25.17,0.491307,2020-03-23,2020-02-19,2020-04-27,68
ONEQ,18.23,-35.23,0.517326,2022-12-28,2021-11-19,2024-01-29,801
SPY,14.89,-33.72,0.441738,2020-03-23,2020-02-19,2020-08-10,173
XLV,12.28,-28.40,0.432176,2020-03-23,2020-01-22,2020-07-15,175
VHT,12.37,-28.85,0.428857,2020-03-23,2020-02-19,2020-06-08,110
GLD,7.14,-42.11,0.169582,2015-12-17,2012-10-04,2020-07-22,2848


### Portfolio drawdowns (default - the first 5 largest)

- `DD` - value of the drawdown (percent)
- `Date` - drawdown date
- `Start` - drawdown starting date
- `End` - drawdown ending date

In [11]:
p4.port_drawdown(fancy=True)

,DD,Date,Start,End,NrDays
No,,,,,
1,-25.17,2020-03-23,2020-02-19,2020-04-27,68
2,-19.02,2022-09-30,2021-12-30,2023-12-13,713
3,-14.10,2018-12-24,2018-10-02,2019-01-31,121
4,-13.29,2025-04-08,2025-02-20,2025-06-26,126
5,-13.01,2015-09-28,2015-07-16,2016-07-08,358


### Portfolio annual (calendar) rate of returns

>Note: the first and last year may not be a full calendar year.

In [12]:
p4.port_annual_returns(fancy=True)

,UnivPort
year,
2015,-5.18%
2016,5.79%
2017,21.63%
2018,4.74%
2019,22.02%
2020,25.91%
2021,17.65%
2022,-12.62%
2023,18.54%


### Portfolio monthly (clandar) rate of returns

In [13]:
p4.port_monthly_returns(fancy=True)

year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
month,,,,,,,,,,,,
1,nan%,-4.97%,3.33%,5.91%,6.35%,0.23%,0.08%,-6.16%,4.06%,1.26%,4.80%,2.94%
2,nan%,2.01%,4.73%,-3.25%,1.84%,-5.47%,-1.18%,-0.44%,-3.57%,3.78%,-0.32%,2.20%
3,nan%,4.02%,0.03%,-0.13%,-2.14%,-1.48%,3.06%,4.48%,4.75%,2.89%,-2.73%,-9.44%
4,nan%,1.92%,1.62%,0.37%,0.69%,12.44%,4.59%,-7.10%,1.72%,-3.13%,-0.26%,4.98%
5,nan%,0.72%,1.05%,1.71%,-3.67%,4.41%,1.74%,-0.50%,-0.57%,3.68%,1.27%,0.15%
6,-1.66%,0.56%,0.28%,-1.94%,6.22%,-2.32%,0.47%,-5.06%,4.58%,2.71%,4.48%,nan%
7,0.85%,4.53%,1.85%,3.26%,0.19%,6.80%,2.96%,5.30%,2.41%,2.25%,0.07%,nan%
8,-5.19%,-1.65%,1.89%,3.33%,0.31%,4.40%,2.36%,-4.58%,-1.36%,2.97%,3.77%,nan%
9,-5.55%,2.02%,1.03%,1.07%,-0.11%,-2.87%,-6.02%,-5.68%,-5.83%,0.94%,4.05%,nan%


### Portfolio returns per reinvestment period 

- `Droll` - rolling date (rebalansing is assumed to be at the closing of the rolling date)
- `Dfinx` - fixing date (the most recent historical date included in the weights computations - last closing date)
- `RR` - portfolio rate of return on the rolling period starting on `Droll`
- *rest of the columns* - prevailing portfolio weights 

In [14]:
p4.port_period_returns(fancy=True)

,Droll,Dfix,RR,VHT,SPY,XLV,GLD,ONEQ
0,2015-06-25,2015-06-24,-8.76,20.00,20.00,20.00,20.00,20.00
1,2015-09-25,2015-09-24,3.03,19.93,19.97,19.93,20.19,19.98
2,2015-12-28,2015-12-24,-0.27,19.97,20.09,20.01,19.85,20.09
3,2016-03-28,2016-03-24,2.17,19.74,20.12,19.84,20.34,19.96
4,2016-06-27,2016-06-24,6.51,19.76,20.04,19.84,20.52,19.84
5,2016-09-27,2016-09-26,-1.15,19.76,20.04,19.81,20.36,20.03
6,2016-12-27,2016-12-23,7.43,19.72,20.34,19.77,19.90,20.27
7,2017-03-28,2017-03-27,4.38,19.74,20.22,19.78,20.00,20.25
8,2017-06-27,2017-06-26,2.12,19.84,20.19,19.86,19.79,20.30
9,2017-09-26,2017-09-25,4.87,19.81,20.20,19.83,19.88,20.29


### Number of shares per portfolio component for each rolling period

In [15]:
p4.get_nshares()

,GLD,ONEQ,SPY,VHT,XLV,_CASH_
Droll,,,,,,
2015-06-25,178,994,95,141,264,0
2015-09-25,167,985,95,142,266,0
2015-12-28,184,969,94,143,264,0
2016-03-28,165,1008,94,153,278,0
2016-06-27,157,1028,95,151,276,0
2016-09-27,165,1004,97,154,284,0
2016-12-27,186,951,91,156,287,0
2017-03-28,181,955,94,155,288,0
2017-06-27,189,936,94,150,280,0


### Other accounting informations

- `Droll` - the start of the rolling period (the end is the next period starting date)
- *portfolio symbols* - number of shares per portfolio component
- `cash_invst` - Dollar equivalent of the shares on the `Droll` date
- `cash_roll` - amount of uninvested cash rolled to the next period. A negative value indicates that the investor need to add this cash amount in order to execute the rolling. The main reasons for these small cash amounts are shares price differential between the fixing (computations) and rolling (execution) dates, as well as rounding to an integer the number of shares.
- `cash_divd` - amount of cash collected form dividend payments during the rolling period (it is an approximation considering the ex-dividend day as the dividend pay day - in practice between these dates could be a gap of a few days or even few weeks).

Note: the value of `cash_roll` can be minimized if,
1. set fixing date to be the same as the rolling date (implies the ability to run the portfolio optimization and execute the rolling transactions close to the end of trading day)
2. a large initial capital will lower, in a relative bases, the impact of rounding to an integer the number of shares.

In [16]:
p4.get_account(fancy=True)

,GLD,ONEQ,SPY,VHT,XLV,_CASH_,cash_invst,cash_roll,cash_divd
Droll,,,,,,,,,
2015-06-25,178.0,994.0,95.0,141.0,264.0,0.0,100108.56,-108.56,0.00
2015-09-25,167.0,985.0,95.0,142.0,266.0,0.0,90224.22,1384.62,377.90
2015-12-28,184.0,969.0,94.0,143.0,264.0,0.0,95317.45,441.72,263.33
2016-03-28,165.0,1008.0,94.0,153.0,278.0,0.0,94531.22,38.67,270.22
2016-06-27,157.0,1028.0,95.0,151.0,276.0,0.0,95055.63,1176.42,300.63
2016-09-27,165.0,1004.0,97.0,154.0,284.0,0.0,104064.59,-433.69,318.73
2016-12-27,186.0,951.0,91.0,156.0,287.0,0.0,101207.54,-311.44,368.05
2017-03-28,181.0,955.0,94.0,155.0,288.0,0.0,108601.55,-176.31,275.03
2017-06-27,189.0,936.0,94.0,150.0,280.0,0.0,112326.15,971.65,260.07


# Universal Portfolio - weights evaluation on a fixing date

This is an example of how to evaluate efficiently the portfolio weights in a fixing date.
It follow a similar procedure as any other portfolio weights evaluation.

We will reuse the `mktdata` previously collected in cell [3].

We star by setting the universal portfolio from a fixing schedule. Later will do a similar computation where the universal portfolio will be set from a hard given fixing date.

### Set a fixing schedule 

For Universal portfolio, the computations of the weights in the fixing date requires the knowledge of all the previous 
fixing dates (as well as market data on these dates). We accomplish this by using `azapy.schedule_simple` function.

- `sdate` - start date of available historical data. The schedule start date will be greater but as close is possible to this date. 
- `edate`- end date of available historical data. The schedule end date is smaller but as close is possible to this date.
- `freq` - fixing frequency. It could be `M` for monthly or `Q` for quarterly.
- `noffset` - rolling date offset - number of business days offset form the last business day of the period (monthly or quarterly).
- `fixoffset` - fixing date offset in business days relative to the rolling date



In [17]:
fixing_schedule = az.schedule_simple(sdate, edate, 
                                     freq='M', noffset=-5, fixoffset=0)
fixing_schedule

,Droll,Dfix
0,2012-01-24,2012-01-24
1,2012-02-22,2012-02-22
2,2012-03-23,2012-03-23
3,2012-04-23,2012-04-23
4,2012-05-23,2012-05-23
...,...,...
168,2026-01-23,2026-01-23
169,2026-02-20,2026-02-20
170,2026-03-24,2026-03-24
171,2026-04-23,2026-04-23


## Set the `azapy.UniversalEngine` class from a fixing schedule

Alternatively, the UniversalEngine object can be set from a hard given fixing date (we will use this approach later in this script).

In [18]:
puniv = az.UniversalEngine(mktdata, schedule=fixing_schedule)

### Compute the portfolio weights

- `mc_paths` - are the number of simulations per batch (must be >= 1)
- `nr_batches` - are the number of batches (must be >= 1). Note that the computation is multithreaded with one batch per thread.
- `variance_reduction = True` - default value (the MC will use the antithetic variance reduce implied by the permutations of the basket components)
- `alpha_dirichlet = None` - default value. In this case a uniform random generator of vectors in the M-simplex is used. This is equivalent to a Flat Dirichlet random generator (all alpha set to 1).
- `mc_seed` - random generator seed value
- `verbose` - print out
    * number of MC simulations (`mc_paths * nr_batches * M!` if `variance_reduction = True` and `mc_paths * nr_batches` otherwise, where `M` is the number of portfolio components and `M!` its factorial).
    * simulation time
    
>Note: if `variance_reduction = True` (the default value) then both the effective number of MC simulations and the computation time grow factorial with the number of portfolio components. For large portfolios it is recommended to the impact of this setup.

>Note: multithreading under Python implementation has reduced effect. In our case we get a time reduction around 30%
   


In [19]:

ww = puniv.getWeights(mc_paths=100, nr_batches=16, mc_seed=42, finalonly=False, verbose=True)

nr simulations: 192000
simulation time: 0.476893


### New portfolio weights

These are in the last raw of `ww`.

>Note: `ww` includes all historical weights (along the fixing schedule) - `finalonly` was set to `False`

In [20]:
ww

symbol,GLD,ONEQ,SPY,VHT,XLV
date,,,,,
2012-01-24,0.200000,0.200000,0.200000,0.200000,0.200000
2012-02-22,0.201024,0.200564,0.199959,0.199372,0.199082
2012-03-23,0.198416,0.201700,0.200611,0.199814,0.199459
2012-04-23,0.198236,0.201023,0.200248,0.200346,0.200146
2012-05-23,0.197746,0.200785,0.200231,0.200747,0.200490
...,...,...,...,...,...
2025-12-23,0.178412,0.219568,0.205125,0.198463,0.198432
2026-01-23,0.180672,0.218593,0.204400,0.198132,0.198205
2026-02-20,0.181459,0.217775,0.204577,0.197967,0.198222


## Set the `azapy.UniversalEngine` class 

### Build the UniversalEngine object

The fixing schedule is build internally going backward from the fixing date with a step of either 21 (for `freq='M'`) or 63 (for `freq='Q'`). The resulting schedule is approximatively equivalent with the one returned by `azapy.shedule_simple` function (that is using a business calendar and in general is more sophisticated).

In [21]:
puniv2 = az.UniversalEngine(mktdata, freq='M')

###  Weights computation - same as before

In [22]:
ww = puniv2.getWeights(mc_paths=100, nr_batches=16, mc_seed=42, verbose=True)
print(ww)

nr simulations: 192000


simulation time: 0.457964
symbol
GLD     0.196523
ONEQ    0.217920
SPY     0.205174
VHT     0.189786
XLV     0.190596
Name: 2026-04-30 00:00:00, dtype: float64


## Set the `azapy.UniversalEngine` class with a Dirichlet random generator

We start by defining the Dirichlet alpha coefficient (must be between 0 and 1)

Here we choose all to be equal to the inverse of the number of portfolio components

In [23]:
dirichlet_alpha = [1 / len(symb)] * len(symb)

### Computation of the weights 

We start by passing the `dirichlet_alpha` coefficients to the constructor.

The rest is the same a above.

>Note: setting all alpha to 1 is equivalent to using a uniform random generator of vectors in the M-simplex of portfolio weights.

In [24]:
puniv2 = az.UniversalEngine(mktdata, freq='M', dirichlet_alpha=dirichlet_alpha)
ww = puniv2.getWeights(mc_paths=100, nr_batches=16, mc_seed=42, verbose=True)
print(ww)

nr simulations: 192000
simulation time: 0.435749
symbol
GLD     0.185432
ONEQ    0.255355
SPY     0.214970
VHT     0.171095
XLV     0.173148
Name: 2026-04-30 00:00:00, dtype: float64
